# Netz Online Stroke HTR (HTR-Baseline-002)

**Policy: HARD LANDING.** Real IAM-OnDB strokes only. No synthetic data, no fallback generators. If the dataset is not attached, the notebook stops with a clear message.

### What changed vs. 001
- **Loader fixed:** IAM-OnDB keeps strokes (`lineStrokes/…/a01-000u-01.xml`) and transcriptions (`ascii/…/a01-000u.txt`, lines after `CSR:`) in separate folders. They are now matched by form id + line number, and mismatches are skipped and reported.
- **Official IAM-OnDB splits** (`trainset / testset_v / testset_t / testset_f`) are used when found, otherwise a split by *form* (never by line) so one page cannot leak across train and test.
- **Sequences up to 1024 points** (was 256) with variable-length batching, packed GRU, and correct CTC `input_lengths`.
- **Height-based normalization** for text lines; stroke-aware subsampling.
- **Augmentation** (rotation, shear, scale, jitter, point thinning), AdamW + OneCycle, AMP, early stopping on **validation CER**, best-checkpoint saving.
- **CER and WER**, ONNX export with PyTorch-vs-ONNX verification, plus a feature test vector for the JS port.

### How to run on Kaggle
1. **File -> Import Notebook** -> `train_htr.ipynb`
2. **IAM-OnDB cannot be auto-downloaded (licence).** One-time: register free at fki.tic.heia-fr.ch, download `lineStrokes-all.tar.gz`, `ascii-all.tar.gz` (+ `trainset.txt`, `testset_v.txt`, `testset_t.txt`, `testset_f.txt`), upload them as a *private* Kaggle dataset, then **+ Add Input** it - or set `NETZ_IAM_KAGGLE=yourname/slug` to have the notebook fetch it itself. Archives are extracted automatically.
3. Accelerator: **GPU T4 x2 / P100**. Click **Run All**.
4. Download `netz_htr.onnx`, `netz_htr_vocab.json`, `netz_htr_feature_test.json` into `Netz/public/models/`.

Optional env overrides: `NETZ_IAM_DIR`, `NETZ_EPOCHS`, `NETZ_MAX_LEN`, `NETZ_MAX_SAMPLES` (smoke tests).

In [ ]:
# 1. Environment & configuration
import os, sys, re, math, json, time, random, hashlib, copy, subprocess
import xml.etree.ElementTree as ET
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def _pip(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
try:
    import onnx, onnxruntime as ort
except ImportError:
    _pip(['onnx', 'onnxruntime'])
    import onnx, onnxruntime as ort

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

CFG = dict(
    max_len=int(os.environ.get('NETZ_MAX_LEN', 1024)),
    epochs=int(os.environ.get('NETZ_EPOCHS', 60)),
    batch_size=int(os.environ.get('NETZ_BATCH', 32)),
    lr=1e-3,
    patience=10,
    min_epochs=int(os.environ.get('NETZ_MIN_EPOCHS', 25)),      # CTC sits on a 'blank' plateau first; never stop before this
    max_samples=int(os.environ.get('NETZ_MAX_SAMPLES', 0)) or None,
    num_workers=int(os.environ.get('NETZ_WORKERS', 2)),
    scale_mode='height',
)
print(f'HTR environment: {device} | PyTorch {torch.__version__} | AMP={use_amp}')
print('Config:', CFG)

# ======================================================================
# Netz canonical stroke features  --  IDENTICAL in all three notebooks.
# Mirror this exactly in the web app (JS) and verify with the exported
# *_feature_test.json file.
#
# Raw ink array: float32 (N, 4) = [x, y, t, stroke_id]
# Feature columns (8): 0 x | 1 y | 2 dx | 3 dy | 4 dt | 5 stroke_end | 6 stroke_start | 7 speed
# ======================================================================
FEATURE_DIM = 8
FEATURE_SPEC = ['x', 'y', 'dx', 'dy', 'dt', 'stroke_end', 'stroke_start', 'speed']


def strokes_to_array(strokes):
    """strokes: list of strokes; a point is dict(x,y[,t]) or a sequence (x, y[, t])."""
    rows = []
    for sid, stroke in enumerate(strokes):
        for pt in stroke:
            if isinstance(pt, dict):
                x, y, t = pt['x'], pt['y'], pt.get('t', None)
            else:
                x, y = pt[0], pt[1]
                t = pt[2] if len(pt) >= 3 else None
            rows.append((float(x), float(y), np.nan if t is None else float(t), float(sid)))
    arr = np.asarray(rows, dtype=np.float64).reshape(-1, 4)
    if len(arr) and np.isfinite(arr[:, 2]).any():
        arr[:, 2] -= np.nanmin(arr[:, 2])          # relative time keeps float32 precise
    return arr.astype(np.float32)


def _to_seconds(t):
    """Auto-detect ms vs s from the median sampling gap; fall back to 10 ms/sample if no timing."""
    t = np.asarray(t, dtype=np.float64)
    n = len(t)
    fallback = np.arange(n, dtype=np.float64) * 0.01
    if n < 2 or not np.isfinite(t).all():
        return fallback
    d = np.diff(t)
    pos = d[d > 0]
    if len(pos) == 0:
        return fallback
    return t / 1000.0 if np.median(pos) > 1.0 else t


def extract_features(arr, max_len, scale_mode='max'):
    """Return float32 (n<=max_len, 8). No padding here (batching pads and tracks real lengths).
    scale_mode: 'max'    -> divide by max(width, height)      (math, router)
                'height' -> divide by max(height, 0.03*width) (text lines)"""
    arr = np.asarray(arr)
    n = len(arr)
    if n == 0:
        return np.zeros((1, FEATURE_DIM), dtype=np.float32)
    t_sec = _to_seconds(arr[:, 2])
    if n > max_len:                                   # deterministic, JS-reproducible subsampling
        idx = np.floor(np.arange(max_len) * (n - 1) / (max_len - 1) + 0.5).astype(np.int64)
        arr, t_sec = arr[idx], t_sec[idx]
        n = max_len
    x = arr[:, 0].astype(np.float64)
    y = arr[:, 1].astype(np.float64)
    sid = arr[:, 3]
    x0, y0 = x.min(), y.min()
    w, h = x.max() - x0, y.max() - y0
    scale = max(h, 0.03 * w, 1e-6) if scale_mode == 'height' else max(w, h, 1e-6)
    xn, yn = (x - x0) / scale, (y - y0) / scale
    dx = np.diff(xn, prepend=xn[0])
    dy = np.diff(yn, prepend=yn[0])
    dt = np.diff(t_sec, prepend=t_sec[0])
    dt = np.where(np.arange(n) == 0, 0.0, np.maximum(dt, 1e-3))
    speed = np.sqrt(dx * dx + dy * dy) / np.maximum(dt, 1e-3)
    start = np.ones(n)
    start[1:] = (sid[1:] != sid[:-1])
    end = np.ones(n)
    end[:-1] = (sid[:-1] != sid[1:])
    feats = np.stack([xn, yn, dx, dy, np.minimum(dt * 10.0, 1.0), end, start, np.minimum(speed / 5.0, 1.0)], axis=1)
    return feats.astype(np.float32)


# ---------------- augmentation on raw ink (train only) ----------------
def thin_points(arr, rng, keep_min=0.7):
    """Randomly drop points (simulates other sampling rates); stroke endpoints are always kept."""
    n = len(arr)
    if n < 8:
        return arr
    sid = arr[:, 3]
    keep = rng.random(n) < rng.uniform(keep_min, 1.0)
    keep[0] = keep[-1] = True
    keep[1:] |= (sid[1:] != sid[:-1])
    keep[:-1] |= (sid[:-1] != sid[1:])
    return arr[keep]


def augment_array(arr, rng, thin=True):
    a = np.array(arr, dtype=np.float64)
    if len(a) == 0:
        return arr
    if thin:
        a = thin_points(a, rng)
    x, y = a[:, 0], a[:, 1]
    cx, cy = x.mean(), y.mean()
    ang = rng.uniform(-0.08, 0.08)
    sx, sy = rng.uniform(0.85, 1.15), rng.uniform(0.85, 1.15)
    shear = rng.uniform(-0.25, 0.25)
    xr, yr = x - cx, y - cy
    x2 = xr * np.cos(ang) - yr * np.sin(ang)
    y2 = xr * np.sin(ang) + yr * np.cos(ang)
    x2 = x2 + shear * y2
    a[:, 0] = x2 * sx + cx
    a[:, 1] = y2 * sy + cy
    span = max(np.ptp(a[:, 0]), np.ptp(a[:, 1]), 1e-6)
    a[:, :2] += rng.normal(0.0, 0.002 * span, size=(len(a), 2))
    return a.astype(np.float32)


# ---------------- shared metrics ----------------
def edit_distance(a, b):
    """Levenshtein distance between two sequences (strings or lists)."""
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


# ---------------- parity test vector for the JS port ----------------
TEST_STROKES = [
    # case 1: tiny, ms timestamps, pen-up gap between the two strokes
    [[[0, 0, 0], [10, 5, 10], [20, 8, 20], [30, 20, 30]],
     [[5, 40, 100], [15, 42, 110], [25, 50, 120]]],
    # case 2: 3000 points -> exercises the subsampling path
    [[[i, round(100 * math.sin(i / 50), 3), i * 10] for i in range(1500)],
     [[i, round(100 * math.cos(i / 50), 3) + 300, 20000 + i * 10] for i in range(1500)]],
]


def write_feature_test(path, max_len, scale_mode):
    cases = []
    for strokes in TEST_STROKES:
        f = extract_features(strokes_to_array(strokes), max_len, scale_mode)
        cases.append({'strokes': strokes, 'features': np.round(f, 6).tolist()})
    with open(path, 'w') as fh:
        json.dump({'feature_spec': FEATURE_SPEC, 'max_len': max_len, 'scale_mode': scale_mode,
                   'tolerance': 1e-4, 'cases': cases}, fh)
    print(f'Wrote {path}  (feed these strokes through the JS extractor; max abs diff must be < 1e-4)')

# ---------------- dataset discovery helpers (archives + diagnostics) ----------------
import tarfile, zipfile
from collections import Counter

def list_inputs(max_dirs=40):
    """Print what is actually attached under /kaggle/input so a failed search is easy to debug."""
    base = '/kaggle/input'
    if not os.path.isdir(base):
        print('No /kaggle/input here (not running on Kaggle?). Set NETZ_IAM_DIR / NETZ_MATH_DIR to your data folders.')
        return
    print('Attached inputs:', sorted(os.listdir(base)) or 'NONE - click "+ Add Input" in the right sidebar')
    shown = 0
    for dp, dn, fn in os.walk(base):
        if dp[len(base):].count(os.sep) > 4:
            dn[:] = []
            continue
        exts = Counter(os.path.splitext(f)[1].lower() or '(none)' for f in fn)
        print(f'  {dp}   subdirs={len(dn)} files={len(fn)} {dict(exts.most_common(4))}')
        shown += 1
        if shown >= max_dirs:
            print('  ...'); break

def extract_archives(src_roots, dest=None):
    """Extract only the ink/transcript files from .zip/.tar/.tar.gz/.tgz found in src_roots (skips 'synthetic')."""
    wanted = ('.inkml', '.xml', '.txt', '.jsonl', '.mlf')
    if dest is None:
        dest = '/kaggle/working/netz_extracted' if os.path.isdir('/kaggle/working') else os.path.expanduser('~/.cache/netz/extracted')
    out = []
    for root in src_roots:
        for dp, _, fn in os.walk(root):
            for f in fn:
                low = f.lower()
                if not low.endswith(('.tar.gz', '.tgz', '.tar', '.zip')):
                    continue
                target = os.path.join(dest, re.sub(r'\W+', '_', f))
                full = os.path.join(dp, f)
                if not os.path.isdir(target):
                    os.makedirs(target, exist_ok=True)
                    print(f'Extracting {full} -> {target} (ink/transcript files only)')
                    try:
                        if low.endswith('.zip'):
                            with zipfile.ZipFile(full) as zf:
                                names = [n for n in zf.namelist() if n.lower().endswith(wanted) and 'synthetic' not in n.lower()]
                                zf.extractall(target, members=names)
                        else:
                            with tarfile.open(full) as tf:
                                members = [m for m in tf if m.isfile() and m.name.lower().endswith(wanted) and 'synthetic' not in m.name.lower()]
                                try:
                                    tf.extractall(target, members=members, filter='data')
                                except TypeError:
                                    tf.extractall(target, members=members)
                    except Exception as e:
                        print('  could not extract:', e)
                out.append(target)
    return out


MATHWRITING_URL = os.environ.get('NETZ_MW_URL', 'https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz')   # official, 2.9 GB

def download_mathwriting(dest='/kaggle/working/mathwriting_real'):
    """Official Google MathWriting (needs Internet: ON). Streams the .tgz and keeps ONLY human-written train/valid/test InkML
    (synthetic and symbols are skipped, which also saves ~GBs of disk). Returns the folder or None."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/mathwriting_real')
    if os.path.isdir(dest) and any(f.endswith('.inkml') for _, _, fs in os.walk(dest) for f in fs):
        return dest
    os.makedirs(dest, exist_ok=True)
    tgz = os.path.join(dest, 'mathwriting.tgz')
    try:
        print(f'Downloading MathWriting from {MATHWRITING_URL} (2.9 GB, takes a few minutes) ...')
        urllib.request.urlretrieve(MATHWRITING_URL, tgz, _progress_hook())
        n = 0
        with tarfile.open(tgz, 'r|gz') as tf:                       # streaming: no seeking, low memory
            for m in tf:
                parts = [p.lower() for p in m.name.split('/')]
                if m.isfile() and m.name.lower().endswith('.inkml') and any(p in ('train', 'valid', 'test') for p in parts[:-1]) \
                        and not any(p in ('synthetic', 'symbols') for p in parts):
                    tf.extract(m, dest, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
                    n += 1
        os.remove(tgz)
        print(f'Extracted {n} human-written InkML files to {dest}')
        return dest if n else None
    except Exception as e:
        print(f'MathWriting download failed: {e}\n  -> turn Internet ON in Settings, or attach MathWriting as an input.')
        return None


def _progress_hook():
    state = {'next': 0}
    def hook(blocks, bsize, total):
        if total > 0:
            pct = min(100, int(blocks * bsize * 100 / total))
            if pct >= state['next']:
                print(f'   download {pct}%', flush=True)
                state['next'] = pct + 10
    return hook

IAM_KAGGLE_HANDLE = os.environ.get('NETZ_IAM_KAGGLE', '')     # e.g. 'yourname/iam-ondb' = YOUR private upload of IAM-OnDB
IAM_URLS = [u for u in os.environ.get('NETZ_IAM_URLS', '').split(',') if u.strip()]   # direct links to lineStrokes-all.tar.gz, ascii-all.tar.gz, ...

def download_iam(dest='/kaggle/working/iam_downloaded'):
    """IAM-OnDB is licence-gated (free registration at fki.tic.heia-fr.ch), so there is NO public auto-download.
    This fetches it from a place YOU control: a private Kaggle dataset (NETZ_IAM_KAGGLE) or direct URLs (NETZ_IAM_URLS)."""
    import urllib.request
    if not os.path.isdir('/kaggle/working'):
        dest = os.path.expanduser('~/.cache/netz/iam_downloaded')
    os.makedirs(dest, exist_ok=True)
    got = []
    if IAM_KAGGLE_HANDLE:
        try:
            import kagglehub
            print(f'Downloading your IAM-OnDB upload from Kaggle: {IAM_KAGGLE_HANDLE}')
            got.append(kagglehub.dataset_download(IAM_KAGGLE_HANDLE))
        except Exception as e:
            print(f'  kagglehub download failed: {e}')
    for url in IAM_URLS:
        try:
            fn = os.path.join(dest, os.path.basename(url.split('?')[0]) or 'iam_file')
            print(f'Downloading {url}')
            urllib.request.urlretrieve(url.strip(), fn, _progress_hook())
            got.append(dest)
        except Exception as e:
            print(f'  download failed: {e}')
    return sorted(set(got))

IAM_HELP = (
    'IAM-OnDB cannot be auto-downloaded (licence: free registration required).\n'
    '  ONE-TIME SETUP, then it is automatic forever:\n'
    '   1. (The old unibe.ch address is retired.) Register (free) at https://fki.tic.heia-fr.ch/databases/iam-on-line-handwriting-database and download the On-Line DB files:\n'
    '      lineStrokes-all.tar.gz, ascii-all.tar.gz and the 4 split files trainset.txt, testset_v.txt, testset_t.txt, testset_f.txt\n'
    '   2. Kaggle -> Datasets -> New Dataset -> upload those files -> keep it PRIVATE.\n'
    '   3. In this notebook either click "+ Add Input" and attach it, OR set  NETZ_IAM_KAGGLE=yourname/your-dataset-slug  (auto-download).')

In [ ]:
# 2. Locate real IAM-OnDB (HARD LANDING: NO FALLBACK) and index it
LINE_XML_RE = re.compile(r'^([a-z]\d{2}-\d{3}[a-z]?)-(\d{2})\.xml$', re.I)   # a01-000u-01.xml
FORM_TXT_RE = re.compile(r'^([a-z]\d{2}-\d{3}[a-z]?)\.txt$', re.I)           # a01-000u.txt
SPLIT_FILES = {'trainset.txt': 'train', 'testset_v.txt': 'val', 'testset_t.txt': 'val', 'testset_f.txt': 'test'}

def read_text(path):
    with open(path, 'r', encoding='latin1') as fh:
        return fh.read()

def read_id_list(path):
    ids = set()
    for line in read_text(path).splitlines():
        tok = os.path.basename(line.strip())
        tok = re.sub(r'\.(xml|txt)$', '', tok, flags=re.I).lower()
        if tok:
            ids.add(tok)
    return ids


MLF_SPACE = {'sp', '<space>', '<sp>', 'space', '|', '_'}
MLF_SYMBOLS = {'pt': '.', 'km': ',', 'sc': ';', 'ga': "'"}          # IAM-OnDB label names: Punkt, Komma, Semikolon, Gaensefuesschen
CONTRACTION_TAILS = {'s', 't', 'll', 'd', 're', 've', 'm'}

def mlf_tokens_to_text(toks):
    """Real IAM-OnDB labels.mlf is character-level: letters, 'sp' = space, and symbol names (pt . / km , / sc ; / ga ').
    Punctuation tokens are written as separate words (always surrounded by 'sp'), so spaces around them are fixed up here."""
    items = [(' ' if t.lower() in MLF_SPACE else MLF_SYMBOLS.get(t, t)) for t in toks]
    out = []
    for k, c in enumerate(items):
        if c == ' ' and k + 1 < len(items) and items[k + 1] in ('.', ',', ';'):
            continue                                                  # no space before . , ;
        out.append(c)
    text = ''.join(out)
    text = re.sub(r"(?<=[A-Za-z]) ' (?=(?:%s)\b)" % '|'.join(CONTRACTION_TAILS), "'", text)   # I ' ve -> I've
    return re.sub(r'\s+', ' ', text).strip()

def parse_mlf(path):
    """HTK master label file -> {line_id: text}. Entries:  "/some/path/a01-000u-01.lab"  <one label per line>  '.'
    Handles character-level labels (IAM-OnDB) and word-level labels."""
    out, cur, toks = {}, None, []
    def flush():
        if cur is None or not toks:
            return
        single = sum(1 for t in toks if len(t) == 1 or t.lower() in MLF_SPACE or t in MLF_SYMBOLS)
        out[cur] = mlf_tokens_to_text(toks) if single >= 0.9 * len(toks) else ' '.join(toks)
    for raw in read_text(path).splitlines():
        line = raw.strip()
        if not line or line.startswith('#!MLF'):
            continue
        if line.startswith('"'):
            flush()
            toks = []
            cur = re.sub(r'\.(lab|rec|txt)$', '', os.path.basename(line.strip('"')), flags=re.I).lower()
        elif line == '.':
            flush()
            cur, toks = None, []
        elif cur is not None:
            parts = line.split()
            lab = parts[2] if len(parts) >= 3 and parts[0].lstrip('-').isdigit() and parts[1].lstrip('-').isdigit() else parts[0]
            toks.append(lab[1:] if len(lab) > 1 and lab[0] == '\\' else lab)       # HTK escapes some symbols with a backslash
    flush()
    return out

mlf_files = []

def scan_iam(extra=()):
    roots = [r for r in [os.environ.get('NETZ_IAM_DIR'), '/kaggle/input', 'data'] if r and os.path.isdir(r)] + list(extra)
    line_xml, form_txt, split_ids = {}, {}, {}
    mlf_files.clear()
    for root in roots:
        for dp, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith('.mlf'):
                    mlf_files.append(os.path.join(dp, f))
                m = LINE_XML_RE.match(f)
                if m:
                    line_xml[(m.group(1).lower(), int(m.group(2)))] = os.path.join(dp, f)
                elif FORM_TXT_RE.match(f):
                    form_txt[f[:-4].lower()] = os.path.join(dp, f)
                elif f in SPLIT_FILES:
                    split_ids.setdefault(SPLIT_FILES[f], set()).update(read_id_list(os.path.join(dp, f)))
    return roots, line_xml, form_txt, split_ids

roots, line_xml, form_txt, split_ids = scan_iam()
if not line_xml:
    roots, line_xml, form_txt, split_ids = scan_iam(extract_archives(roots))     # maybe the data is zipped
if not line_xml:
    dl = download_iam()                                                           # YOUR private Kaggle dataset / URLs (optional)
    if dl:
        roots, line_xml, form_txt, split_ids = scan_iam(dl + extract_archives(dl))
if not line_xml:
    list_inputs()
    listing = []
    for r in roots:
        for dp, dn, fn in os.walk(r):
            listing.append(f'{dp}  ({len(fn)} files)')
            if len(listing) > 25: break
    raise FileNotFoundError(
        '\n' + '=' * 70 + '\n[FATAL - HARD LANDING]\nNo IAM-OnDB line stroke files (e.g. a01-000u-01.xml) found.\n'
        + IAM_HELP + '\nSearched: ' + str(roots) + '\nFirst folders seen:\n  ' + '\n  '.join(listing) + '\n' + '=' * 70)

print(f'[Real Data Confirmed] {len(line_xml)} line-stroke files, {len(form_txt)} form transcriptions, '
      f'official split lists: {sorted(split_ids)}')

In [ ]:
# 3. Vocabulary, text cleaning, transcription matching
chars = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,!?:;'-()\"/"
char2idx = {c: i + 1 for i, c in enumerate(chars)}      # 0 = CTC blank
idx2char = {i + 1: c for i, c in enumerate(chars)}
num_classes = len(chars) + 1

def clean_text(t):
    t = t.replace('&quot;', '"').replace('&amp;', '&').replace('&apos;', "'")
    t = re.sub(r'\s+', ' ', t).strip()
    if not t:
        return None
    bad = sum(1 for c in t if c not in char2idx)
    if bad > 0.1 * len(t):
        return None
    t = ''.join(c for c in t if c in char2idx)
    return re.sub(r'\s+', ' ', t).strip() or None

def csr_lines(path):
    lines = [l.strip() for l in read_text(path).splitlines()]
    if 'CSR:' not in lines:
        return []
    return [l for l in lines[lines.index('CSR:') + 1:] if l]

def line_id_of(form, idx):
    return f'{form}-{idx:02d}'

mlf_text = {}
for mp in mlf_files:
    mlf_text.update(parse_mlf(mp))
if mlf_text:
    print(f'Read {len(mlf_text)} transcriptions from {len(mlf_files)} .mlf file(s). Examples:')
    for k in list(mlf_text)[:3]:
        print(f'   {k}: "{mlf_text[k]}"')

lines_per_form = {}
for (form, idx) in line_xml:
    lines_per_form[form] = lines_per_form.get(form, 0) + 1
csr_cache = {}
records, skipped = [], {'no_transcript': 0, 'count_mismatch': 0, 'bad_text': 0}
for (form, idx), xml_path in sorted(line_xml.items()):
    text = None
    paired = os.path.splitext(xml_path)[0] + '.txt'
    if os.path.exists(paired):                                   # layout: txt next to xml
        text = read_text(paired).strip()
    elif form in form_txt:                                       # layout: IAM-OnDB ascii/ folder
        if form not in csr_cache:
            csr_cache[form] = csr_lines(form_txt[form])
        lines = csr_cache[form]
        if len(lines) != lines_per_form[form]:
            skipped['count_mismatch'] += 1
            continue
        text = lines[idx - 1]
    elif line_id_of(form, idx) in mlf_text:                      # layout: labels.mlf (HTK master label file)
        text = mlf_text[line_id_of(form, idx)]
    else:
        skipped['no_transcript'] += 1
        continue
    text = clean_text(text)
    if text is None:
        skipped['bad_text'] += 1
        continue
    records.append(dict(form=form, line_id=line_id_of(form, idx), xml=xml_path, text=text))

print(f'Matched {len(records)} lines with transcriptions. Skipped: {skipped}')
_used = ''.join(sorted({c for r in records for c in r['text']}))
print('Characters present in the labels:', repr(_used))
if not any(c.isdigit() for c in _used):
    print('NOTE: the labels contain no digits, so the model cannot learn to write numbers (the router handles NUMBER separately).')
if not any(c in _used for c in '-()?:!"/'):
    print('NOTE: only a reduced punctuation set is present in these labels; the model will only output what it saw.')
if len(records) == 0 or len(records) < 0.5 * len(line_xml):
    raise RuntimeError('Too few stroke files could be matched to transcriptions - the dataset layout is not what '
                       'the loader expects. Print the folder tree and adapt scan_iam()/csr_lines().')

In [ ]:
# 4. Splits: official IAM-OnDB lists if usable, else split by FORM (no page leaks between splits)
def hash_split(key):
    v = int(hashlib.md5(key.encode()).hexdigest(), 16) % 100
    return 'train' if v < 80 else ('val' if v < 90 else 'test')

def official_split(rec):
    for name in ('train', 'val', 'test'):
        ids = split_ids.get(name, set())
        if rec['line_id'] in ids or rec['form'] in ids:
            return name
    return None

use_official = all(k in split_ids for k in ('train', 'val', 'test'))
if use_official:
    labels = [official_split(r) for r in records]
    counts = {n: labels.count(n) for n in ('train', 'val', 'test')}
    use_official = all(c > 0 for c in counts.values())
if use_official:
    for r, l in zip(records, labels):
        r['split'] = l
    records = [r for r in records if r['split'] is not None]
    print('Using OFFICIAL IAM-OnDB splits:', counts)
else:
    for r in records:
        r['split'] = hash_split(r['form'])
    print('Official split lists not usable -> splitting by form id. NOTE: the same writer may still appear in '
          'several splits, so CER is optimistic compared to the official writer-independent protocol.')

rng_cap = random.Random(SEED)
if CFG['max_samples']:
    rng_cap.shuffle(records)
    records = records[:CFG['max_samples']]

In [ ]:
# 5. Load stroke data into memory
def parse_iam_xml(path):
    root = ET.parse(path).getroot()
    strokes = []
    for st in root.iter('Stroke'):
        pts = [(float(p.get('x')), float(p.get('y')), float(p.get('time', 'nan'))) for p in st.iter('Point')]
        if pts:
            strokes.append(pts)
    return strokes

def ctc_need(text):
    return len(text) + sum(1 for a, b in zip(text, text[1:]) if a == b)

items = {'train': [], 'val': [], 'test': []}
unreadable, infeasible = 0, 0
t0 = time.time()
for i, r in enumerate(records):
    try:
        arr = strokes_to_array(parse_iam_xml(r['xml']))
    except Exception:
        unreadable += 1
        continue
    if len(arr) < 2:
        unreadable += 1
        continue
    if r['split'] == 'train':      # CTC needs enough frames; keep a 30% safety margin for thinning augmentation
        frames = (min(len(arr), CFG['max_len']) + 1) // 2
        if frames < 1.3 * ctc_need(r['text']):
            infeasible += 1
            continue
    items[r['split']].append((arr, r['text']))
    if (i + 1) % 2000 == 0:
        print(f'  parsed {i + 1}/{len(records)} ({time.time() - t0:.0f}s)')
print({k: len(v) for k, v in items.items()}, f'| unreadable={unreadable} | train samples dropped (too few CTC frames)={infeasible}')
for k in ('train', 'val', 'test'):
    if len(items[k]) == 0:
        raise RuntimeError(f'Split "{k}" is empty - cannot train/evaluate honestly.')

class IAMLines(Dataset):
    def __init__(self, data, train):
        self.data, self.train, self.rng = data, train, None
    def __len__(self):
        return len(self.data)
    def __getitem__(self, i):
        arr, text = self.data[i]
        if self.train:
            if self.rng is None:
                self.rng = np.random.default_rng()
            arr = augment_array(arr, self.rng)
        feats = extract_features(arr, CFG['max_len'], CFG['scale_mode'])
        tgt = torch.tensor([char2idx[c] for c in text], dtype=torch.long)
        return torch.from_numpy(feats), tgt, text

def collate(batch):
    feats, tgts, texts = zip(*batch)
    lengths = torch.tensor([len(f) for f in feats], dtype=torch.long)
    x = torch.zeros(len(feats), int(lengths.max()), FEATURE_DIM)
    for i, f in enumerate(feats):
        x[i, :len(f)] = f
    tl = torch.tensor([len(t) for t in tgts], dtype=torch.long)
    return x, lengths, torch.cat(tgts), tl, list(texts)

def make_loader(split, shuffle):
    return DataLoader(IAMLines(items[split], train=(split == 'train')), batch_size=CFG['batch_size'], shuffle=shuffle,
                      collate_fn=collate, num_workers=CFG['num_workers'], drop_last=(split == 'train'))

train_loader, val_loader, test_loader = make_loader('train', True), make_loader('val', False), make_loader('test', False)

In [ ]:
# 6. Sanity check on real samples (mandatory)
print('--- [Sanity check] ---')
ds_check = IAMLines(items['train'], train=False)
lens, tl_ = [], []
for i in range(min(200, len(ds_check))):
    f, t, txt = ds_check[i]
    assert f.ndim == 2 and f.shape[1] == FEATURE_DIM, f.shape
    assert torch.isfinite(f).all(), 'non-finite features'
    assert 0 < len(t) == len(txt)
    lens.append(len(f)); tl_.append(len(txt))
print(f'points/line: mean={np.mean(lens):.0f} p95={np.percentile(lens, 95):.0f} max={max(lens)} | chars/line: mean={np.mean(tl_):.0f}')
print('frames/char ratio (needs >> 1 for CTC):', round(np.mean([(min(l, CFG["max_len"]) + 1) // 2 / max(1, c) for l, c in zip(lens, tl_)]), 2))
for i in range(3):
    print(f'  sample {i}: "{ds_check[i][2]}"')
print('--- [Sanity check passed] ---')

In [ ]:
# 7. Model: stroke projection -> 2x conv (stride 2) -> 3-layer BiGRU -> CTC head
class OnlineStrokeHTR(nn.Module):
    def __init__(self, in_features=FEATURE_DIM, num_classes=num_classes, d=96, conv_ch=128,
                 rnn_hidden=192, rnn_layers=3, dropout=0.2):
        super().__init__()
        self.in_proj = nn.Sequential(nn.Linear(in_features, d), nn.LayerNorm(d), nn.GELU())
        self.conv = nn.Sequential(
            nn.Conv1d(d, conv_ch, 5, stride=2, padding=2), nn.BatchNorm1d(conv_ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(conv_ch, conv_ch, 3, padding=1), nn.BatchNorm1d(conv_ch), nn.GELU(), nn.Dropout(dropout))
        self.rnn = nn.GRU(conv_ch, rnn_hidden, num_layers=rnn_layers, batch_first=True,
                          bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(2 * rnn_hidden, num_classes)

    @staticmethod
    def out_lengths(lengths):
        return (lengths + 1) // 2            # conv1: k=5, s=2, p=2 ; conv2 keeps length

    def forward(self, x, lengths=None):
        h = self.conv(self.in_proj(x).transpose(1, 2)).transpose(1, 2)
        if lengths is not None:               # training/eval with padding: pack so GRU ignores pad frames
            packed = nn.utils.rnn.pack_padded_sequence(h, self.out_lengths(lengths).cpu(), batch_first=True, enforce_sorted=False)
            out, _ = self.rnn(packed)
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=h.size(1))
        else:                                 # inference / ONNX (single unpadded sequence)
            out, _ = self.rnn(h)
        return self.fc(out)

    @staticmethod
    def decode_greedy(logits, out_lens, blank=0):
        preds = logits.argmax(-1).cpu().tolist()
        res = []
        for row, L in zip(preds, out_lens.cpu().tolist()):
            toks, prev = [], None
            for p in row[:L]:
                if p != blank and p != prev:
                    toks.append(p)
                prev = p
            res.append(toks)
        return res

model = OnlineStrokeHTR().to(device)
print(f'Model initialised: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters, {num_classes} classes (blank=0)')

In [ ]:
# 8. Training: CTC + AdamW + OneCycle + AMP, early stopping on validation CER
def evaluate(loader):
    model.eval()
    preds, refs = [], []
    with torch.no_grad():
        for x, lengths, _, _, texts in loader:
            logits = model(x.to(device), lengths)
            for ids in model.decode_greedy(logits, model.out_lengths(lengths)):
                preds.append(''.join(idx2char.get(i, '') for i in ids))
            refs.extend(texts)
    cer = sum(edit_distance(p, r) for p, r in zip(preds, refs)) / max(1, sum(len(r) for r in refs))
    wer = sum(edit_distance(p.split(), r.split()) for p, r in zip(preds, refs)) / max(1, sum(len(r.split()) for r in refs))
    return cer, wer, preds, refs

ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-2)
steps_per_epoch = max(1, len(train_loader))
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG['lr'], total_steps=CFG['epochs'] * steps_per_epoch, pct_start=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_cer, best_state, bad_epochs = float('inf'), None, 0
for epoch in range(1, CFG['epochs'] + 1):
    model.train()
    total, nb, t0 = 0.0, 0, time.time()
    for x, lengths, targets, tlens, _ in train_loader:
        x, targets = x.to(device), targets.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            logits = model(x, lengths)
        log_probs = logits.float().log_softmax(2).transpose(0, 1)
        loss = ctc_loss(log_probs, targets, model.out_lengths(lengths).to(device), tlens.to(device))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item(); nb += 1
    val_cer, val_wer, _, _ = evaluate(val_loader)
    flag = ''
    if val_cer < best_cer:
        best_cer, best_state, bad_epochs, flag = val_cer, copy.deepcopy(model.state_dict()), 0, '  <- best'
    else:
        bad_epochs += 1
    print(f'Epoch {epoch:02d}/{CFG["epochs"]} | train CTC {total / max(1, nb):.3f} | val CER {val_cer * 100:.2f}% | val WER {val_wer * 100:.2f}% | {time.time() - t0:.0f}s{flag}')
    if epoch >= CFG['min_epochs'] and bad_epochs >= CFG['patience']:
        print(f'Early stopping (no val-CER improvement for {CFG["patience"]} epochs).')
        break
model.load_state_dict(best_state)

In [ ]:
# 9. Untouched test-set evaluation (best checkpoint)
test_cer, test_wer, test_preds, test_refs = evaluate(test_loader)
print('=== Untouched Test Set ===')
print(f'Test CER: {test_cer * 100:.2f}% | WER: {test_wer * 100:.2f}% | {len(test_refs)} lines | (best val CER {best_cer * 100:.2f}%)')
for r, p in list(zip(test_refs, test_preds))[:5]:
    print(f'  REF : {r}\n  PRED: {p}\n')
if test_cer > 0.35:
    print('WARNING: CER > 35% - not production quality. Train longer, check the data, or enlarge the model.')

In [ ]:
# 10. Production ONNX + vocab export, verified against PyTorch
model.eval().cpu()
dummy = torch.randn(1, 256, FEATURE_DIM, dtype=torch.float32)
onnx_path = 'netz_htr.onnx'
export_kwargs = dict(export_params=True, opset_version=17, do_constant_folding=True, input_names=['strokes'], output_names=['logits'],
                     dynamic_axes={'strokes': {0: 'batch_size', 1: 'sequence_length'}, 'logits': {0: 'batch_size', 1: 'time_steps'}})
try:
    torch.onnx.export(model, dummy, onnx_path, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(model, dummy, onnx_path, **export_kwargs)
onnx.checker.check_model(onnx.load(onnx_path))

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
max_diff = 0.0
for T in (64, 300, CFG['max_len']):
    xin = torch.randn(1, T, FEATURE_DIM)
    with torch.no_grad():
        ref = model(xin).numpy()
    out = sess.run(None, {'strokes': xin.numpy()})[0]
    max_diff = max(max_diff, float(np.abs(ref - out).max()))
print(f'ONNX vs PyTorch max |logit diff| over several sequence lengths: {max_diff:.2e}')
assert max_diff < 1e-3, 'ONNX export does not match PyTorch'

# real-sample parity (single, unpadded)
arr, text = items['test'][0]
f = torch.from_numpy(extract_features(arr, CFG['max_len'], CFG['scale_mode']))[None]
onnx_logits = sess.run(None, {'strokes': f.numpy()})[0]
ids = model.decode_greedy(torch.from_numpy(onnx_logits), torch.tensor([onnx_logits.shape[1]]))[0]
print(f'ONNX decode of a real test line: "{"".join(idx2char[i] for i in ids)}"   (ref: "{text}")')

with open('netz_htr_vocab.json', 'w') as fh:
    json.dump(list(chars), fh, indent=2)           # index i -> chars[i-1]; index 0 = CTC blank
with open('netz_htr_meta.json', 'w') as fh:
    json.dump({'max_len': CFG['max_len'], 'scale_mode': CFG['scale_mode'], 'blank_index': 0, 'feature_spec': FEATURE_SPEC,
               'test_cer': round(test_cer, 4), 'test_wer': round(test_wer, 4)}, fh, indent=2)
write_feature_test('netz_htr_feature_test.json', CFG['max_len'], CFG['scale_mode'])
print(f'Exported {onnx_path} ({os.path.getsize(onnx_path) / 1024:.1f} KB), netz_htr_vocab.json, netz_htr_meta.json')